# Computerized Adaptive Testing for Efficient LLM Evaluation

This tutorial demonstrates how to use **Computerized Adaptive Testing (CAT)** to efficiently evaluate language models using a fraction of the full benchmark, based on the methodology from:

> Truong, S., Tu, Y., Liang, P., Li, B., & Koyejo, S. (2025). *Reliable and Efficient Amortized Model-based Evaluation.* arXiv:2503.13335.

## Motivation

Evaluating LLMs on large benchmarks (e.g., 13K+ questions in MMLU) is expensive. Not all questions are equally informative — easy questions tell us little about strong models, and hard questions tell us little about weak ones. **Item Response Theory (IRT)** models this relationship, and **CAT** exploits it to select the most informative questions adaptively.

The key finding from the paper: **IRT-based adaptive testing can reduce query complexity to ~53% on average (and up to 86%) while maintaining 95% reliability.**

### What you'll learn

1. Loading real HELM benchmark data with the native `torch_measure` data loader
2. Fitting IRT models to calibrate question difficulty
3. Understanding Fisher information for item selection
4. Running CAT with different strategies (Fisher, Spanning, Random)
5. Evaluating CAT efficiency: fewer questions, same reliability

## 1. Setup

In [ ]:
# pip install torch-measure[data]  # uncomment to install (includes huggingface_hub)

import torch
import matplotlib.pyplot as plt
import numpy as np

from torch_measure.models import Rasch, TwoPL
from torch_measure.cat import AdaptiveTester, fisher_information
from torch_measure.cat import MaxInfoStrategy, SpanningStrategy, RandomStrategy
from torch_measure.data import ResponseMatrix
from torch_measure.datasets import load, list_datasets, info

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"
torch.manual_seed(42)

print("Available HELM datasets:")
for name in list_datasets(family="helm"):
    ds = info(name)
    print(f"  {name:30s}  {ds.n_subjects:>4d} models x {ds.n_items:>6d} items")

## 2. Loading HELM Benchmark Data

We'll work with **MMLU** (Massive Multitask Language Understanding) — one of the most widely used LLM benchmarks, covering 57 knowledge domains with 13,223 multiple-choice questions evaluated across 183 language models.

The `load()` function downloads the data from HuggingFace Hub and returns a `ResponseMatrix` with metadata.

In [ ]:
# Load MMLU benchmark data
rm = load("helm/mmlu")

print(rm)
print(f"Response matrix shape: {rm.shape}")
print(f"Density (fraction observed): {rm.density:.2%}")
print(f"\nSubject (LLM) scores: min={rm.subject_means.min():.3f}, "
      f"max={rm.subject_means.max():.3f}, mean={rm.subject_means.mean():.3f}")
print(f"Item facility (easiness): min={rm.item_means.min():.3f}, "
      f"max={rm.item_means.max():.3f}, mean={rm.item_means.mean():.3f}")

In [ ]:
# Inspect the LLM metadata
if rm.subject_ids is not None:
    print(f"Number of LLMs: {len(rm.subject_ids)}")
    print(f"\nFirst 10 LLMs:")
    for i, sid in enumerate(rm.subject_ids[:10]):
        score = rm.subject_means[i].item()
        meta = rm.subject_metadata[i] if rm.subject_metadata else {}
        org = meta.get("org", "unknown") if meta else "unknown"
        print(f"  {sid:50s}  score={score:.3f}  org={org}")

In [ ]:
# Visualize the response matrix sorted by LLM score and item difficulty
responses = rm.data
n_subjects, n_items = rm.n_subjects, rm.n_items

# Handle NaN for sorting
subject_scores = rm.subject_means
item_facility = rm.item_means

sorted_subj = subject_scores.argsort()
sorted_item = item_facility.argsort(descending=True)  # hardest first

# Subsample for visualization (full matrix is too large)
vis_items = sorted_item[::max(1, n_items // 200)]  # ~200 items evenly spaced
sorted_responses = responses[sorted_subj][:, vis_items]

fig, ax = plt.subplots(figsize=(12, 5))
# Replace NaN with 0.5 for visualization
vis_data = sorted_responses.clone()
vis_data[torch.isnan(vis_data)] = 0.5
ax.imshow(vis_data.numpy(), aspect="auto", cmap="RdYlGn", interpolation="nearest", vmin=0, vmax=1)
ax.set_xlabel("Items (sorted by difficulty, hardest → easiest)")
ax.set_ylabel("LLMs (sorted by ability, weakest → strongest)")
ax.set_title("MMLU Response Matrix (green=correct, red=incorrect, yellow=missing)")
plt.tight_layout()
plt.show()

The Guttman pattern is visible: stronger LLMs (top) answer more questions correctly, and easier items (right) are answered correctly by more models.

In [ ]:
# Distribution of LLM scores and item difficulties
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(subject_scores.numpy(), bins=30, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("Average Score")
axes[0].set_ylabel("Number of LLMs")
axes[0].set_title(f"LLM Score Distribution (n={n_subjects})")
axes[0].axvline(subject_scores.mean().item(), color="red", linestyle="--", label=f"Mean={subject_scores.mean():.2f}")
axes[0].legend()

axes[1].hist(item_facility.numpy(), bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_xlabel("Item Facility (fraction correct)")
axes[1].set_ylabel("Number of Items")
axes[1].set_title(f"Item Difficulty Distribution (n={n_items})")
axes[1].axvline(item_facility.mean().item(), color="red", linestyle="--", label=f"Mean={item_facility.mean():.2f}")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Calibrating the Item Bank with IRT

Before running CAT, we need to **calibrate** the item bank — estimate the difficulty (and optionally discrimination) of each question. This is the foundation that makes adaptive testing possible.

The **Rasch model** defines:

$$P(\text{correct} \mid \theta, b) = \sigma(\theta - b)$$

where $\theta$ is the LLM's ability and $b$ is the question's difficulty.

The paper found that the **Rasch model works well** for LLM evaluation because the limited number of test-takers (LLMs) makes overfitting a concern with more complex models.

In [ ]:
# Fit Rasch model on the full response matrix
rasch = Rasch(n_subjects=n_subjects, n_items=n_items)
history = rasch.fit(responses, max_epochs=500, lr=0.01, verbose=False)

print(f"Final loss: {history['losses'][-1]:.4f}")
print(f"\nEstimated abilities:  min={rasch.ability.min():.2f}, max={rasch.ability.max():.2f}")
print(f"Estimated difficulties: min={rasch.difficulty.min():.2f}, max={rasch.difficulty.max():.2f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(history["losses"])
ax.set_xlabel("Epoch")
ax.set_ylabel("Negative Log-Likelihood")
ax.set_title("Rasch Model Training")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the calibrated item bank
difficulty = rasch.difficulty.detach()
ability = rasch.ability.detach()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Item difficulty distribution
axes[0].hist(difficulty.numpy(), bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[0].set_xlabel("IRT Difficulty (b)")
axes[0].set_ylabel("Count")
axes[0].set_title("Calibrated Item Difficulties")
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.5)

# Ability vs raw score
axes[1].scatter(subject_scores.numpy(), ability.numpy(), alpha=0.6, s=20, color="steelblue")
axes[1].set_xlabel("Raw Average Score")
axes[1].set_ylabel("IRT Ability (θ)")
axes[1].set_title("IRT Ability vs Raw Score")
axes[1].grid(True, alpha=0.3)
r = torch.corrcoef(torch.stack([subject_scores, ability]))[0, 1]
axes[1].annotate(f"r = {r:.3f}", xy=(0.05, 0.9), xycoords="axes fraction", fontsize=11)

plt.tight_layout()
plt.show()

print("IRT ability is a monotonic transform of raw score, but accounts for item difficulty.")
print("Two LLMs with the same raw score on different question subsets can have different abilities.")

## 4. Fisher Information: Why Some Questions Are More Informative

The core idea behind adaptive testing is **Fisher information**. For the Rasch model:

$$I(\theta; b_j) = P_j(\theta) \cdot (1 - P_j(\theta))$$

This is maximized when $\theta = b_j$ (ability matches difficulty), giving $I = 0.25$. In other words, **questions at your ability level are the most informative**.

- A very easy question ($P \approx 1$) tells us little — we already know the model will get it right.
- A very hard question ($P \approx 0$) also tells us little — we already know the model will get it wrong.
- A question where $P \approx 0.5$ provides maximum information about the model's ability.

In [ ]:
# Demonstrate Fisher information
theta_range = torch.linspace(-4, 4, 200)

# Pick three items with different difficulties
sorted_diff, sorted_idx = difficulty.sort()
easy_idx = sorted_idx[n_items // 10]        # 10th percentile
medium_idx = sorted_idx[n_items // 2]        # median
hard_idx = sorted_idx[int(n_items * 0.9)]    # 90th percentile

example_items = [easy_idx.item(), medium_idx.item(), hard_idx.item()]
example_labels = ["Easy", "Medium", "Hard"]
example_colors = ["green", "orange", "red"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Item Characteristic Curves
for idx, label, color in zip(example_items, example_labels, example_colors):
    b = difficulty[idx].item()
    p = torch.sigmoid(theta_range - b)
    axes[0].plot(theta_range.numpy(), p.numpy(), label=f"{label} (b={b:.1f})", color=color, linewidth=2)

axes[0].set_xlabel("Ability (θ)")
axes[0].set_ylabel("P(correct)")
axes[0].set_title("Item Characteristic Curves")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0.5, color="gray", linestyle=":", alpha=0.5)

# Fisher Information Curves
for idx, label, color in zip(example_items, example_labels, example_colors):
    b = difficulty[idx]
    info_vals = fisher_information(theta_range, b.unsqueeze(0)).squeeze(1)
    axes[1].plot(theta_range.numpy(), info_vals.numpy(), label=f"{label} (b={b:.1f})", color=color, linewidth=2)

axes[1].set_xlabel("Ability (θ)")
axes[1].set_ylabel("Fisher Information")
axes[1].set_title("Item Information Curves")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0.25, color="gray", linestyle=":", alpha=0.5, label="Max info = 0.25")

plt.tight_layout()
plt.show()

print("Each item provides maximum information when ability matches its difficulty.")
print("CAT selects items whose difficulty is closest to the current ability estimate.")

In [ ]:
# Total test information function for the entire item bank
theta_range = torch.linspace(-4, 4, 200)
all_info = fisher_information(theta_range, difficulty)  # (200, n_items)
total_info = all_info.sum(dim=1)  # sum across all items

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(theta_range.numpy(), total_info.numpy(), color="steelblue", linewidth=2)
ax.fill_between(theta_range.numpy(), total_info.numpy(), alpha=0.2, color="steelblue")
ax.set_xlabel("Ability (θ)")
ax.set_ylabel("Total Test Information")
ax.set_title(f"Test Information Function (MMLU, {n_items} items)")
ax.grid(True, alpha=0.3)

# Mark where each LLM falls
for i, a in enumerate(ability):
    ax.axvline(a.item(), color="red", alpha=0.1, linewidth=0.5)

ax.set_xlim(-4, 4)
plt.tight_layout()
plt.show()

print("The test information function shows where the benchmark is most precise.")
print("Red lines mark the estimated abilities of the 183 LLMs.")

## 5. Running Computerized Adaptive Testing

Now we run CAT on individual LLMs. The algorithm:

1. **Initialize** ability estimate at $\hat{\theta} = 0$
2. **Select** the most informative unanswered question (maximizing Fisher information at $\hat{\theta}$)
3. **Administer** the question and observe the response
4. **Update** $\hat{\theta}$ via maximum likelihood: $\hat{\theta} = \arg\max_\theta \sum_{j=1}^{t} \log P(y_j \mid \theta, b_j)$
5. **Repeat** until budget is exhausted

### Item Selection Strategies

- **Fisher (MaxInfo)**: Always picks the item maximizing $I(\hat{\theta}; b_j) = P_j(1 - P_j)$
- **Spanning**: First samples items across the difficulty range for a rough estimate, then switches to Fisher
- **Random**: Baseline — picks items randomly

In [ ]:
# Run CAT on a single LLM to see how it works
# Pick an LLM with moderate ability
median_idx = ability.argsort()[n_subjects // 2].item()
subject_name = rm.subject_ids[median_idx] if rm.subject_ids else f"LLM #{median_idx}"
subject_responses = responses[median_idx]
true_ability = ability[median_idx].item()

print(f"Testing LLM: {subject_name}")
print(f"Full-test ability estimate: {true_ability:.3f}")
print(f"Full-test raw score: {rm.subject_means[median_idx]:.3f}")

budget = 100  # Only 100 out of 13,223 questions

# Run all three strategies
torch.manual_seed(42)
tester_fisher = AdaptiveTester(rasch, strategy="fisher")
result_fisher = tester_fisher.run(subject_responses, budget=budget, lr=0.3, n_steps=30)

torch.manual_seed(42)
tester_spanning = AdaptiveTester(rasch, strategy="spanning", n_spanning=20)
result_spanning = tester_spanning.run(subject_responses, budget=budget, lr=0.3, n_steps=30)

torch.manual_seed(42)
tester_random = AdaptiveTester(rasch, strategy="random")
result_random = tester_random.run(subject_responses, budget=budget, lr=0.3, n_steps=30)

print(f"\nAbility estimates with {budget} items (out of {n_items}):")
print(f"  Fisher:   {result_fisher['ability']:.3f}  (error: {abs(result_fisher['ability'] - true_ability):.3f})")
print(f"  Spanning: {result_spanning['ability']:.3f}  (error: {abs(result_spanning['ability'] - true_ability):.3f})")
print(f"  Random:   {result_random['ability']:.3f}  (error: {abs(result_random['ability'] - true_ability):.3f})")

In [ ]:
# Plot ability estimation trajectory
fig, ax = plt.subplots(figsize=(10, 5))
items_range = range(1, budget + 1)

ax.plot(items_range, result_fisher["ability_trajectory"], "-", 
        label="Fisher (max info)", color="steelblue", linewidth=2)
ax.plot(items_range, result_spanning["ability_trajectory"], "-", 
        label="Spanning → Fisher", color="orange", linewidth=2)
ax.plot(items_range, result_random["ability_trajectory"], "-", 
        label="Random", color="gray", linewidth=1.5, alpha=0.7)
ax.axhline(y=true_ability, color="red", linestyle="--", linewidth=1.5,
           label=f"Full-test estimate (θ={true_ability:.2f})")

ax.set_xlabel("Number of Items Administered")
ax.set_ylabel("Estimated Ability (θ)")
ax.set_title(f"CAT Ability Estimation Trajectory — {subject_name}")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Fisher information converges fastest to the full-test ability estimate.")
print(f"Items used: {budget}/{n_items} = {budget/n_items:.1%} of the full benchmark.")

In [ ]:
# Visualize which items each strategy selected
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, result, name in zip(axes, 
    [result_fisher, result_spanning, result_random],
    ["Fisher", "Spanning", "Random"]):
    
    selected_diffs = difficulty[result["administered"]].numpy()
    order = range(len(selected_diffs))
    colors = ["green" if r == 1 else "red" for r in result["responses"]]
    
    ax.scatter(order, selected_diffs, c=colors, s=8, alpha=0.6)
    ax.axhline(true_ability, color="blue", linestyle="--", linewidth=1, label=f"θ={true_ability:.1f}")
    ax.set_xlabel("Administration Order")
    ax.set_title(f"{name} Strategy")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Item Difficulty (b)")
fig.suptitle("Items Selected by Each Strategy (green=correct, red=incorrect)", y=1.02)
plt.tight_layout()
plt.show()

print("Fisher concentrates around the model's ability level (maximum information).")
print("Spanning first explores the full difficulty range, then focuses.")
print("Random samples uniformly — many items are too easy or too hard to be informative.")

## 6. Evaluating CAT Across All LLMs

To properly evaluate CAT efficiency, we run it on **all 183 LLMs** and compare the CAT ability estimates against the full-test estimates (which use all 13K+ questions).

In [ ]:
# Run CAT on all LLMs with varying budgets
budgets = [20, 50, 100, 200, 500]
strategies = ["fisher", "spanning", "random"]
full_ability = rasch.ability.detach()

# Store results: {strategy: {budget: [ability_estimates]}}
all_results = {s: {b: [] for b in budgets} for s in strategies}

for i in range(n_subjects):
    subject_resp = responses[i]
    # Skip subjects with too many missing responses
    if torch.isnan(subject_resp).sum() > n_items * 0.5:
        for s in strategies:
            for b in budgets:
                all_results[s][b].append(float("nan"))
        continue
    
    for s in strategies:
        for b in budgets:
            torch.manual_seed(i * 1000 + b)
            tester = AdaptiveTester(rasch, strategy=s, n_spanning=min(20, b // 2))
            result = tester.run(subject_resp, budget=b, lr=0.3, n_steps=30)
            all_results[s][b].append(result["ability"])

print("CAT evaluation complete.")
print(f"Tested {n_subjects} LLMs x {len(budgets)} budgets x {len(strategies)} strategies")

In [ ]:
# Compute correlation with full-test ability for each strategy and budget
print(f"{'Strategy':<12} {'Budget':>8} {'Correlation':>12} {'MAE':>8} {'% of items':>12}")
print("-" * 56)

correlations = {s: [] for s in strategies}

for s in strategies:
    for b in budgets:
        cat_est = torch.tensor(all_results[s][b])
        # Filter out NaN
        valid = ~torch.isnan(cat_est) & ~torch.isnan(full_ability)
        if valid.sum() < 10:
            correlations[s].append(float("nan"))
            continue
        corr = torch.corrcoef(torch.stack([cat_est[valid], full_ability[valid]]))[0, 1].item()
        mae = (cat_est[valid] - full_ability[valid]).abs().mean().item()
        correlations[s].append(corr)
        print(f"{s:<12} {b:>8} {corr:>12.4f} {mae:>8.3f} {b/n_items:>11.1%}")

In [ ]:
# Plot correlation vs budget for each strategy
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {"fisher": "steelblue", "spanning": "orange", "random": "gray"}
labels = {"fisher": "Fisher (max info)", "spanning": "Spanning → Fisher", "random": "Random"}

for s in strategies:
    valid_corrs = [c for c in correlations[s] if not (isinstance(c, float) and c != c)]
    valid_budgets = [b for b, c in zip(budgets, correlations[s]) if not (isinstance(c, float) and c != c)]
    axes[0].plot(valid_budgets, valid_corrs, "o-", color=colors[s], 
                 label=labels[s], linewidth=2, markersize=6)

axes[0].set_xlabel("Number of Items (Budget)")
axes[0].set_ylabel("Correlation with Full-Test Ability")
axes[0].set_title("CAT Efficiency: Correlation vs Budget")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0.95, color="red", linestyle=":", alpha=0.5, label="r=0.95")
axes[0].set_ylim(0.5, 1.02)

# Scatter: CAT estimate vs full-test estimate (best budget)
best_budget = 100
cat_fisher = torch.tensor(all_results["fisher"][best_budget])
valid = ~torch.isnan(cat_fisher) & ~torch.isnan(full_ability)
axes[1].scatter(full_ability[valid].numpy(), cat_fisher[valid].numpy(), 
                alpha=0.5, s=20, color="steelblue")
lims = [full_ability[valid].min().item() - 0.5, full_ability[valid].max().item() + 0.5]
axes[1].plot(lims, lims, "k--", alpha=0.3)
axes[1].set_xlabel("Full-Test Ability (all items)")
axes[1].set_ylabel(f"CAT Ability ({best_budget} items)")
corr_val = torch.corrcoef(torch.stack([full_ability[valid], cat_fisher[valid]]))[0, 1].item()
axes[1].set_title(f"Fisher CAT ({best_budget} items, {best_budget/n_items:.1%}) vs Full Test (r={corr_val:.3f})")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Empirical Reliability

Following the paper, we compute **empirical reliability** — a measure of how stable the ability estimates are. It is defined as:

$$R = 1 - \frac{\sum_i I(\hat{\theta}_i)^{-1}}{\sum_i (\hat{\theta}_i - \bar{\theta})^2}$$

where $I(\hat{\theta}_i)$ is the total Fisher information at the estimated ability, summed over administered items. Higher reliability ($R \to 1$) means more precise estimation.

In [ ]:
def empirical_reliability(ability_estimates, difficulty, administered_items_list):
    """Compute empirical reliability of CAT estimates.
    
    Parameters
    ----------
    ability_estimates : torch.Tensor
        Estimated abilities (N,).
    difficulty : torch.Tensor
        Item difficulties (M,).
    administered_items_list : list[list[int]]
        Items administered to each subject.
    
    Returns
    -------
    float
        Empirical reliability in [0, 1].
    """
    theta_bar = ability_estimates.mean()
    variance = ((ability_estimates - theta_bar) ** 2).sum()
    
    total_se_sq = 0.0
    for i, items in enumerate(administered_items_list):
        theta_i = ability_estimates[i]
        item_diffs = difficulty[items]
        info_i = fisher_information(theta_i, item_diffs).sum()
        total_se_sq += 1.0 / info_i.clamp(min=1e-6)
    
    return (1 - total_se_sq / variance.clamp(min=1e-6)).item()


# Compute reliability at each budget for Fisher strategy
print(f"{'Budget':>8} {'% Items':>10} {'Reliability':>12}")
print("-" * 34)

# Re-run Fisher CAT storing administered items
reliability_results = []
for budget in budgets:
    administered_lists = []
    cat_abilities = []
    for i in range(n_subjects):
        subject_resp = responses[i]
        if torch.isnan(subject_resp).sum() > n_items * 0.5:
            continue
        torch.manual_seed(i * 1000 + budget)
        tester = AdaptiveTester(rasch, strategy="fisher")
        result = tester.run(subject_resp, budget=budget, lr=0.3, n_steps=30)
        administered_lists.append(result["administered"])
        cat_abilities.append(result["ability"])
    
    abilities_t = torch.tensor(cat_abilities)
    rel = empirical_reliability(abilities_t, difficulty, administered_lists)
    reliability_results.append(rel)
    print(f"{budget:>8} {budget/n_items:>9.1%} {rel:>12.4f}")

In [ ]:
# Plot reliability vs budget
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(budgets, reliability_results, "o-", color="steelblue", linewidth=2, markersize=8)
ax.axhline(0.95, color="red", linestyle="--", alpha=0.7, label="95% reliability threshold")
ax.set_xlabel("Number of Items (Budget)")
ax.set_ylabel("Empirical Reliability")
ax.set_title("CAT Reliability vs Test Length (MMLU)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 1.02)

# Add percentage labels
for b, r in zip(budgets, reliability_results):
    ax.annotate(f"{b/n_items:.1%}", (b, r), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=9)

plt.tight_layout()
plt.show()

# Find budget for 95% reliability
for b, r in zip(budgets, reliability_results):
    if r >= 0.95:
        print(f"\n95% reliability achieved with {b} items ({b/n_items:.1%} of the full benchmark).")
        break
else:
    print(f"\n95% reliability not achieved in tested budgets. Max reliability: {max(reliability_results):.4f}")

## 8. Why IRT Outperforms Raw Scoring

A key insight from the paper: raw average scores are **sensitive to which subset of questions is used**. If you evaluate on 50 easy questions, all models look good. If you evaluate on 50 hard questions, all models look bad.

IRT ability estimates are **robust** because they account for question difficulty.

In [ ]:
# Demonstrate IRT robustness vs raw scoring on different difficulty subsets
torch.manual_seed(42)

sorted_diff_idx = difficulty.argsort()
n_subset = 200

easy_items = sorted_diff_idx[:n_subset]       # easiest 200 items
hard_items = sorted_diff_idx[-n_subset:]      # hardest 200 items
random_items = sorted_diff_idx[torch.randperm(n_items)[:n_subset]]  # random 200

# Raw scores on each subset
def raw_scores(data, items):
    subset = data[:, items]
    obs = ~torch.isnan(subset)
    subset_clean = subset.clone()
    subset_clean[~obs] = 0
    return (subset_clean.sum(dim=1) / obs.float().sum(dim=1).clamp(min=1))

easy_scores = raw_scores(responses, easy_items)
hard_scores = raw_scores(responses, hard_items)
random_scores = raw_scores(responses, random_items)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, scores, label, color in zip(axes, 
    [easy_scores, hard_scores, random_scores],
    ["Easy Items", "Hard Items", "Random Items"],
    ["green", "red", "gray"]):
    
    valid = ~torch.isnan(scores) & ~torch.isnan(full_ability)
    ax.scatter(full_ability[valid].numpy(), scores[valid].numpy(), alpha=0.5, s=20, color=color)
    r = torch.corrcoef(torch.stack([full_ability[valid], scores[valid]]))[0, 1].item()
    ax.set_xlabel("IRT Ability (θ, full test)")
    ax.set_ylabel(f"Raw Score ({label})")
    ax.set_title(f"{label} (n={n_subset}, r={r:.3f})")
    ax.grid(True, alpha=0.3)

plt.suptitle("Raw Scores Are Sensitive to Item Subset — IRT Is Not", y=1.03, fontsize=13)
plt.tight_layout()
plt.show()

print("On easy items, raw scores cluster near 1.0 with poor discrimination.")
print("On hard items, raw scores cluster near 0.0 with poor discrimination.")
print("IRT ability accounts for difficulty, providing consistent rankings regardless of subset.")

## 9. CAT Across Multiple Benchmarks

The paper evaluated CAT across 22 HELM benchmarks. Let's run CAT on a few representative ones to see how efficiency varies by benchmark characteristics.

In [ ]:
# Compare CAT efficiency across benchmarks
benchmark_names = ["helm/mmlu", "helm/gsm8k", "helm/truthfulqa", "helm/boolq", "helm/commonsense"]
budget_frac = 0.1  # Use 10% of items

print(f"{'Benchmark':<25} {'Items':>7} {'Budget':>7} {'Corr (Fisher)':>14} {'Corr (Random)':>14}")
print("-" * 72)

bench_results = []

for bname in benchmark_names:
    # Load benchmark
    brm = load(bname)
    bdata = brm.data
    bn_subj, bn_items = brm.n_subjects, brm.n_items
    bbudget = max(20, int(bn_items * budget_frac))
    
    # Fit Rasch model
    bmodel = Rasch(n_subjects=bn_subj, n_items=bn_items)
    bmodel.fit(bdata, max_epochs=300, lr=0.01, verbose=False)
    bfull_ability = bmodel.ability.detach()
    bdiff = bmodel.difficulty.detach()
    
    # Run CAT (Fisher and Random)
    fisher_ests, random_ests = [], []
    for i in range(bn_subj):
        resp = bdata[i]
        if torch.isnan(resp).sum() > bn_items * 0.5:
            fisher_ests.append(float("nan"))
            random_ests.append(float("nan"))
            continue
        
        torch.manual_seed(i * 1000)
        r_f = AdaptiveTester(bmodel, strategy="fisher").run(resp, budget=bbudget, lr=0.3, n_steps=30)
        fisher_ests.append(r_f["ability"])
        
        torch.manual_seed(i * 1000)
        r_r = AdaptiveTester(bmodel, strategy="random").run(resp, budget=bbudget, lr=0.3, n_steps=30)
        random_ests.append(r_r["ability"])
    
    fisher_t = torch.tensor(fisher_ests)
    random_t = torch.tensor(random_ests)
    valid = ~torch.isnan(fisher_t) & ~torch.isnan(bfull_ability)
    
    corr_f = torch.corrcoef(torch.stack([fisher_t[valid], bfull_ability[valid]]))[0, 1].item()
    corr_r = torch.corrcoef(torch.stack([random_t[valid], bfull_ability[valid]]))[0, 1].item()
    
    bench_results.append({"name": bname, "n_items": bn_items, "budget": bbudget,
                          "corr_fisher": corr_f, "corr_random": corr_r})
    
    print(f"{bname:<25} {bn_items:>7} {bbudget:>7} {corr_f:>14.4f} {corr_r:>14.4f}")

In [ ]:
# Visualize cross-benchmark results
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(bench_results))
width = 0.35
ax.bar([i - width/2 for i in x], [r["corr_fisher"] for r in bench_results], 
       width, label="Fisher (adaptive)", color="steelblue")
ax.bar([i + width/2 for i in x], [r["corr_random"] for r in bench_results], 
       width, label="Random (baseline)", color="gray", alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels([r["name"].split("/")[1] for r in bench_results], rotation=15)
ax.set_ylabel("Correlation with Full-Test Ability")
ax.set_title("CAT Efficiency Across HELM Benchmarks (10% of items)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0.5, 1.05)

# Add item count annotations
for i, r in enumerate(bench_results):
    ax.annotate(f"{r['budget']}/{r['n_items']}", (i, 0.52), 
                ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.show()

## 10. CAT with 2PL Model

The 2PL model adds a **discrimination** parameter $a_j$ per item:

$$P(\text{correct} \mid \theta, a_j, b_j) = \sigma(a_j(\theta - b_j))$$

Items with higher discrimination are steeper (better at differentiating ability levels). The Fisher information becomes:

$$I(\theta; a_j, b_j) = a_j^2 \cdot P_j \cdot (1 - P_j)$$

High-discrimination items are **quadratically** more informative — CAT preferentially selects them.

In [ ]:
# Fit 2PL model and run CAT
twopl = TwoPL(n_subjects=n_subjects, n_items=n_items)
history_2pl = twopl.fit(responses, max_epochs=500, lr=0.01, verbose=False)

disc = twopl.discrimination.detach()
diff_2pl = twopl.difficulty.detach()
ability_2pl = twopl.ability.detach()

print(f"2PL discrimination — mean: {disc.mean():.2f}, std: {disc.std():.2f}, "
      f"range: [{disc.min():.2f}, {disc.max():.2f}]")

# Run CAT with 2PL model
budget_2pl = 100
cat_fisher_2pl = []
cat_fisher_1pl = []

for i in range(n_subjects):
    subject_resp = responses[i]
    if torch.isnan(subject_resp).sum() > n_items * 0.5:
        cat_fisher_2pl.append(float("nan"))
        cat_fisher_1pl.append(float("nan"))
        continue
    
    torch.manual_seed(i * 1000 + budget_2pl)
    r2 = AdaptiveTester(twopl, strategy="fisher").run(subject_resp, budget=budget_2pl, lr=0.3, n_steps=30)
    cat_fisher_2pl.append(r2["ability"])
    
    torch.manual_seed(i * 1000 + budget_2pl)
    r1 = AdaptiveTester(rasch, strategy="fisher").run(subject_resp, budget=budget_2pl, lr=0.3, n_steps=30)
    cat_fisher_1pl.append(r1["ability"])

cat_2pl_t = torch.tensor(cat_fisher_2pl)
cat_1pl_t = torch.tensor(cat_fisher_1pl)
valid = ~torch.isnan(cat_2pl_t) & ~torch.isnan(full_ability)

corr_2pl = torch.corrcoef(torch.stack([cat_2pl_t[valid], full_ability[valid]]))[0, 1].item()
corr_1pl = torch.corrcoef(torch.stack([cat_1pl_t[valid], full_ability[valid]]))[0, 1].item()

print(f"\nCorrelation with full-test (budget={budget_2pl}):")
print(f"  Rasch (1PL) CAT: {corr_1pl:.4f}")
print(f"  2PL CAT:         {corr_2pl:.4f}")

In [ ]:
# Visualize discrimination values and their effect on item information
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Discrimination distribution
axes[0].hist(disc.numpy(), bins=40, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("Discrimination (a)")
axes[0].set_ylabel("Count")
axes[0].set_title("2PL Discrimination Distribution")
axes[0].axvline(1.0, color="red", linestyle="--", alpha=0.5, label="Rasch (a=1)")
axes[0].legend()

# Max information per item: a^2 * 0.25
max_info_per_item = disc ** 2 * 0.25
axes[1].scatter(disc.numpy(), max_info_per_item.numpy(), alpha=0.3, s=10, color="steelblue")
axes[1].set_xlabel("Discrimination (a)")
axes[1].set_ylabel("Maximum Fisher Information")
axes[1].set_title("Higher Discrimination → More Informative Items")
axes[1].axhline(0.25, color="red", linestyle="--", alpha=0.5, label="Rasch max (0.25)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This tutorial demonstrated the CAT methodology from [Truong et al. (2025)](https://arxiv.org/abs/2503.13335) using `torch_measure`:

| Step | What | API |
|------|------|-----|
| **Data Loading** | Load HELM benchmarks with metadata | `datasets.load("helm/mmlu")` |
| **IRT Calibration** | Estimate item difficulty & LLM ability | `Rasch.fit()`, `TwoPL.fit()` |
| **Fisher Information** | Quantify item informativeness at each ability level | `fisher_information()` |
| **Adaptive Testing** | Select maximally informative items | `AdaptiveTester(strategy="fisher")` |
| **Efficiency** | Evaluate correlation & reliability vs test length | Empirical reliability metric |

### Key Takeaways

1. **CAT reduces evaluation cost dramatically** — Fisher-based item selection achieves high correlation with full-test estimates using a fraction of the items
2. **IRT > raw scores** — IRT ability estimates are robust to which subset of questions is used, while raw scores are sensitive to subset difficulty
3. **Fisher information guides selection** — items where P(correct) ≈ 0.5 at the current ability estimate are most informative
4. **Spanning initialization helps** — sampling across the difficulty range first gives a better initial ability estimate

### References

- Truong, S., Tu, Y., Liang, P., Li, B., & Koyejo, S. (2025). *Reliable and Efficient Amortized Model-based Evaluation.* [arXiv:2503.13335](https://arxiv.org/abs/2503.13335)
- Liang, P., et al. (2023). *Holistic Evaluation of Language Models.* TMLR.